In [ ]:
# --- Environment Setup (do not edit) ---
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401

        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True


PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab": PROJECT_ROOT / ".env.colab",
    "vastai": PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac": PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    print("\u26a0\ufe0f  WARNING: Could not detect platform. Falling back to .env (local).")
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    print(
        f"\u26a0\ufe0f  WARNING: Detected /workspace but VAST_CONTAINERLABEL is not set. Assuming Vast.ai."
    )
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv

if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    print(f"\u26a0\ufe0f  WARNING: Expected env file not found: {_env_file}")
    if _fallback.exists():
        print(f"   Falling back to: {_fallback}")
        _env_file = _fallback
    else:
        raise FileNotFoundError(
            f"No .env file found. Copy the correct example file:\n"
            f"  cp .env.{PLATFORM or 'mac'}.example .env.{PLATFORM or 'mac'}"
        )

load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()

print(f"\u2705 Platform : {PLATFORM or 'unknown (local fallback)'}")
print(f"\u2705 Env file : {_env_file}")
print(f"\u2705 DATA_ROOT: {DATA_ROOT}")
print(f"{'\u2705' if DATA_ROOT.exists() else '\u274c'} Path exists: {DATA_ROOT.exists()}")
# ---------------------------------------------------------------------------

# ViFactCheck Gold Evidence Training (Stage 3.9a)

Fine-tunes **`vinai/phobert-large`** on ViFactCheck using **Statement + Evidence** (gold evidence) as the sentence pair.

Per paper §4.3, the **Gold Evidence** mode outperforms Full Context mode. This notebook follows the same
original architecture and training setup as `plm_training.py`, but swaps `Context` → `Evidence` as the second sentence.

- **Model**: `vinai/phobert-large` (1024-dim hidden)
- **Architecture**: Pooler output → Dropout(0.3) → Linear (original)
- **Input**: `Statement + Evidence` (gold evidence — best mode per paper §4.3)
- **Loss**: Plain `CrossEntropyLoss()`
- **Optimizer**: `AdamW(lr=5e-6)` — no scheduler, no warmup
- **Training**: Fixed 10 epochs, no early stopping
- **Classification**: 3-class (Supported=0, Refuted=1, NEI=2)

```
Output: training/checkpoints_vifactcheck_gold/<run>/best_model.pth + tokenizer/
```

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)

CONFIG = {
    "paths": {
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_vifactcheck_gold",
    },
    "data": {
        "hf_dataset": "tranthaihoa/vifactcheck",
        "text_field": "Statement",
        "context_field": "Evidence",  # Gold evidence (best mode per paper §4.3)
        "label_field": "labels",
        "max_length": 256,
        "num_classes": 3,
    },
    "model": {
        "backbone": "vinai/phobert-large",
        "dropout": 0.3,
        "num_classes": 3,
    },
    "training": {
        "batch_size": 16,
        "epochs": 10,
        "lr": 0.5e-5,
        "seed": 28,
    },
    "safety": {
        "smoke_test": False,
        "smoke_batches": 2,
    },
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Checkpoint:   {CONFIG['paths']['checkpoint_root']}")
print(f"Backbone:     {CONFIG['model']['backbone']}")
print(f"Input mode:   Statement + Evidence (gold evidence)")

In [ ]:
# --- DEPENDENCY PREFLIGHT ---
import importlib, sys

_required = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
}

_missing = []
for mod, pkg in _required.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        _missing.append(pkg)

if _missing:
    print(f"Missing packages: {_missing}")
    raise RuntimeError(f"Missing packages: {_missing}.")
else:
    print("All dependencies satisfied.")

In [ ]:
# --- IMPORTS AND SETUP ---
import os, gc, json, random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from transformers import AutoTokenizer, AutoModel

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG["paths"]["checkpoint_root"].mkdir(parents=True, exist_ok=True)

print(f"PyTorch      : {torch.__version__}")
import transformers

print(f"Transformers : {transformers.__version__}")


def select_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"Device: cuda ({torch.cuda.get_device_name(0)}, {mem:.1f} GB)")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("Device: mps (Apple Silicon). For full training, prefer a CUDA GPU.")
    else:
        dev = torch.device("cpu")
        print("Device: cpu. Use smoke_test=True for local validation.")
    return dev


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Seed: {seed}")


DEVICE = select_device()
seed_everything(CONFIG["training"]["seed"])

## Step 1: Load ViFactCheck Dataset (Gold Evidence Mode)

Uses `Statement + Evidence` as sentence pairs. Labels: 0=Supported, 1=Refuted, 2=NEI.

In [ ]:
from datasets import load_dataset

hf_name = CONFIG["data"]["hf_dataset"]
text_field = CONFIG["data"]["text_field"]
ctx_field = CONFIG["data"]["context_field"]
label_field = CONFIG["data"]["label_field"]

print(f"Loading '{hf_name}' from HuggingFace...")
raw = load_dataset(hf_name)

available = list(raw.keys())
print(f"  Available splits: {available}")
sample = dict(raw[available[0]][0])
print(f"  Columns: {list(sample.keys())}")

print(f"\n  text_field    = {text_field!r}  (claim)")
print(f"  context_field = {ctx_field!r}  (gold evidence -- best mode per paper §4.3)")
print(f"  label_field   = {label_field!r}")

raw_examples = {}
for hf_split in available:
    ds = raw[hf_split]
    examples = []
    for item in ds:
        stmt = str(item.get(text_field) or "").strip()
        evi = str(item.get(ctx_field) or "").strip()
        label = int(item.get(label_field))
        if not stmt:
            continue
        examples.append({"text": stmt, "text_b": evi, "label": label})
    nc = CONFIG["data"]["num_classes"]
    hist = [sum(1 for e in examples if e["label"] == i) for i in range(nc)]
    print(f"  {hf_split:5s}: {len(examples):5d} examples | label dist {hist}")
    raw_examples[hf_split] = examples

print(f"\nData loaded: { {k: len(v) for k, v in raw_examples.items()} }")

## Step 2: Tokenize and Build DataLoaders

In [ ]:
class SentencePairDataset(Dataset):
    def __init__(self, sentence_pairs, labels, tokenizer, max_length):
        self.sentence_pairs = sentence_pairs
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentence_pairs)

    def __getitem__(self, idx):
        sentence1, sentence2 = self.sentence_pairs[idx]
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            sentence1,
            text_pair=sentence2,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding="max_length",
            return_attention_mask=True,
            return_tensors="pt",
            truncation=True,
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "label": torch.tensor(label, dtype=torch.long),
        }


backbone = CONFIG["model"]["backbone"]
print(f"Loading tokenizer: {backbone}")
tokenizer = AutoTokenizer.from_pretrained(backbone)
print("Tokenizer loaded.")

_max_len = CONFIG["data"]["max_length"]
_bs = CONFIG["training"]["batch_size"]
_smoke = CONFIG["safety"]["smoke_test"]
_smoke_n = CONFIG["safety"]["smoke_batches"]

datasets_map = {}
for split, exs in raw_examples.items():
    pairs = [(e["text"], e["text_b"]) for e in exs]
    labels = [e["label"] for e in exs]
    datasets_map[split] = SentencePairDataset(pairs, labels, tokenizer, _max_len)


def _make_loader(ds, shuffle, batch_size):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


loaders = {
    "train": _make_loader(datasets_map["train"], shuffle=True, batch_size=_bs),
    "dev": _make_loader(datasets_map.get("dev", datasets_map["train"]), shuffle=False, batch_size=_bs),
    "test": _make_loader(datasets_map.get("test", datasets_map["train"]), shuffle=False, batch_size=_bs),
}

if _smoke:
    from itertools import islice

    class _SmokeLoader:
        def __init__(self, loader, n):
            self._l = loader
            self._n = n

        def __iter__(self):
            return islice(self._l, self._n)

        def __len__(self):
            return min(self._n, len(self._l))

    loaders = {k: _SmokeLoader(v, _smoke_n) for k, v in loaders.items()}
    print(f"SMOKE TEST: {_smoke_n} batches per split")

_b = next(iter(loaders["train"]))
print(
    f"\nBatch shapes -- input_ids: {tuple(_b['input_ids'].shape)}  "
    f"attention_mask: {tuple(_b['attention_mask'].shape)}  "
    f"label: {tuple(_b['label'].shape)}"
)
for split in ["train", "dev", "test"]:
    _ds = datasets_map.get(split)
    if _ds:
        nc = CONFIG["data"]["num_classes"]
        hist = [sum(1 for l in _ds.labels if l == i) for i in range(nc)]
        print(f"  {split:5s}: {len(_ds):5d} samples | label dist {hist}")

## Step 3: Build PhoBERT Classifier (Original Architecture)

In [ ]:
class PhoBERTClassifier(nn.Module):
    def __init__(self, phobert, num_classes):
        super(PhoBERTClassifier, self).__init__()
        self.phobert = phobert
        self.dropout = nn.Dropout(0.3)
        self.linear = nn.Linear(self.phobert.config.hidden_size, num_classes)
        self.hidden_size = self.phobert.config.hidden_size

    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.phobert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False,
        )
        dropout_output = self.dropout(pooled_output)
        logits = self.linear(dropout_output)
        return logits


print(f"Building model: {backbone}")
phobert = AutoModel.from_pretrained(backbone)
model = PhoBERTClassifier(phobert, num_classes=CONFIG["model"]["num_classes"]).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters -- total: {total:,}  trainable: {trainable:,}")
print(f"Hidden size: {model.hidden_size}")

model.eval()
with torch.no_grad():
    _ids = _b["input_ids"][:2].to(DEVICE)
    _mask = _b["attention_mask"][:2].to(DEVICE)
    _logit = model(_ids, _mask)
print(f"Sanity forward output shape: {tuple(_logit.shape)}")

## Step 4: Loss and Optimizer (Original)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["training"]["lr"])

print(f"Loss      : CrossEntropyLoss (plain)")
print(f"Optimizer : AdamW  lr={CONFIG['training']['lr']}")
print(f"Epochs    : {CONFIG['training']['epochs']}")

## Step 5: Run Directory

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"vifactcheck_gold_{timestamp}"

run_dir = CONFIG["paths"]["checkpoint_root"] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {run_dir}")

## Step 6: Training Loop

In [ ]:
def compute_metrics(y_true, y_pred, nc):
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    per_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    return {"accuracy": round(acc, 4), "macro_f1": round(macro_f1, 4),
            "per_f1": [round(float(f), 4) for f in per_f1]}


best_val_f1 = -1.0
best_epoch = -1
history = []
best_ckpt_path = run_dir / "best_model.pth"

epochs = CONFIG["training"]["epochs"]

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    n_batches = 0
    pbar = tqdm(loaders["train"], desc=f"Epoch {epoch+1}/{epochs} [train]", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    mean_train_loss = train_loss / max(1, n_batches)

    model.eval()
    predictions, true_labels = [], []
    pbar = tqdm(loaders["dev"], desc=f"Epoch {epoch+1}/{epochs} [dev]", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask)
            _, predicted = torch.max(outputs, 1)
            predictions.extend(predicted.cpu().numpy().tolist())
            true_labels.extend(labels.cpu().numpy().tolist())

    dev_metrics = compute_metrics(true_labels, predictions, CONFIG["data"]["num_classes"])
    print(f"Epoch {epoch+1}/{epochs}  train_loss={mean_train_loss:.4f}  "
          f"dev_acc={dev_metrics['accuracy']:.4f}  dev_macro_f1={dev_metrics['macro_f1']:.4f}")

    history.append({
        "epoch": epoch + 1,
        "train_loss": round(mean_train_loss, 4),
        "dev_accuracy": dev_metrics["accuracy"],
        "dev_macro_f1": dev_metrics["macro_f1"],
    })

    if dev_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = dev_metrics["macro_f1"]
        best_epoch = epoch + 1
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": best_epoch,
            "dev_metrics": dev_metrics,
            "backbone": CONFIG["model"]["backbone"],
            "hidden_size": model.hidden_size,
            "num_classes": CONFIG["model"]["num_classes"],
        }, best_ckpt_path)
        print(f"  -> New best (dev_macro_f1={best_val_f1:.4f}), saved")

print(f"\nTraining complete. Best epoch: {best_epoch}  dev_macro_f1: {best_val_f1:.4f}")

## Step 7: Final Test Evaluation

In [ ]:
if not best_ckpt_path.exists():
    print("best_model.pth not found -- skipping test evaluation.")
else:
    _ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(_ckpt["model_state_dict"])
    print(f"Loaded best checkpoint (epoch {_ckpt['epoch']}) for test evaluation.")

    model.eval()
    predictions, true_labels = [], []
    pbar = tqdm(loaders["test"], desc="Test", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask)
            _, predicted = torch.max(outputs, 1)
            predictions.extend(predicted.cpu().numpy().tolist())
            true_labels.extend(labels.cpu().numpy().tolist())

    test_metrics = compute_metrics(true_labels, predictions, CONFIG["data"]["num_classes"])
    print(f"\nTest Results:")
    print(f"  Accuracy  : {test_metrics['accuracy']:.4f}")
    print(f"  Macro F1  : {test_metrics['macro_f1']:.4f}")
    print(f"  Per-class F1: {test_metrics['per_f1']}")
    print()
    print(classification_report(true_labels, predictions, digits=4))

    report_path = run_dir / "test_report.json"
    with open(report_path, "w") as f:
        json.dump({"test_metrics": test_metrics, "best_epoch": best_epoch,
                   "best_val_f1": best_val_f1}, f, indent=2)
    print(f"Test report saved: {report_path}")

## Step 8: Visualize Training Curves and Confusion Matrix

In [ ]:
if not history:
    print("No training history to plot.")
else:
    sns.set_theme(style="whitegrid", palette="muted")
    hist_df = pd.DataFrame(history)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(hist_df["epoch"], hist_df["train_loss"], marker="o", color="steelblue")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].plot(hist_df["epoch"], hist_df["dev_accuracy"], marker="s", color="seagreen")
    axes[1].axvline(x=best_epoch, color="red", linestyle="--", alpha=0.5, label=f"Best (epoch {best_epoch})")
    axes[1].set_title("Dev Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    axes[2].plot(hist_df["epoch"], hist_df["dev_macro_f1"], marker="^", color="coral")
    axes[2].axvline(x=best_epoch, color="red", linestyle="--", alpha=0.5, label=f"Best (epoch {best_epoch})")
    axes[2].set_title("Dev Macro F1")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Macro F1")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(run_dir / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

    if "true_labels" in dir() and "predictions" in dir():
        cm = confusion_matrix(true_labels, predictions)
        class_names = ["Supported", "Refuted", "NEI"]
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_title("Test Confusion Matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        plt.tight_layout()
        plt.savefig(run_dir / "confusion_matrix.png", dpi=150, bbox_inches="tight")
        plt.show()

## Step 9: Save Tokenizer and Manifest

In [ ]:
tokenizer_dir = run_dir / "tokenizer"
tokenizer.save_pretrained(str(tokenizer_dir))
print(f"Tokenizer saved: {tokenizer_dir}")

manifest = {
    "best_checkpoint_path": str(best_ckpt_path),
    "tokenizer_path": str(tokenizer_dir),
    "best_epoch": best_epoch,
    "best_dev_macro_f1": best_val_f1,
    "backbone": CONFIG["model"]["backbone"],
    "hidden_size": model.hidden_size,
    "num_classes": CONFIG["model"]["num_classes"],
    "input_format": "Statement + Evidence (gold evidence)",
    "architecture": "pooler_output -> Dropout(0.3) -> Linear",
    "training_setup": {
        "loss": "CrossEntropyLoss (plain)",
        "optimizer": "AdamW",
        "lr": CONFIG["training"]["lr"],
        "epochs": CONFIG["training"]["epochs"],
        "batch_size": CONFIG["training"]["batch_size"],
        "max_length": CONFIG["data"]["max_length"],
    },
    "data_source": "HuggingFace tranthaihoa/vifactcheck",
    "label_scheme": "3-class: Supported(0), Refuted(1), NEI(2)",
    "note": "Gold Evidence mode (paper §4.3 best mode), follows original plm_training.py",
}
if "test_metrics" in dir():
    manifest["test_metrics"] = test_metrics

manifest_path = run_dir / "checkpoint_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest written: {manifest_path}")
print(f"\nDone. All outputs in: {run_dir}")